# 02 — Versioning, Reproducibility, dan CI/CD untuk ML

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/riki-profile/ai-engineer-journey/blob/main/13-mlops/02_versioning_reproducibility_cicd.ipynb)

Sebuah hasil ML **reproducible** jika orang lain (atau kamu sendiri 6 bulan lagi) bisa mendapatkan hasil yang sama dari nol. Untuk itu, **semua** bahan yang memengaruhi model harus tercatat:

| Bahan | Bisa berubah karena | Cara mengunci |
|---|---|---|
| **Kode** | Commit baru | Git (catat hash commit di setiap run) |
| **Data** | Dataset diperbarui, dibersihkan, ditambah | Hash file / DVC, validasi data |
| **Environment** | Versi library berubah | Lock file, Docker |
| **Keacakan** | Inisialisasi bobot, urutan data | Seed |
| **Hardware** | CPU vs GPU, versi CUDA | Catat; terima selisih kecil |

Isi notebook:
1. Keacakan dan **variasi antar-seed**.
2. **Versioning data** dengan hash dan validasi data.
3. **Lineage**: mencatat kode, data, dan environment di setiap run MLflow.
4. Manajemen **environment**.
5. **Uji perilaku** model: yang tidak terlihat dari akurasi.
6. **CI/CD untuk ML** dan gerbang kualitas (*quality gate*) di repo ini.

## 0. Instalasi dan Kode

In [ ]:
%pip install -q mlflow torch pandas onnx onnxscript onnxruntime pytest

Modul ini memakai kembali model sentimen dari modul 12 ([`12-deployment/app/`](../12-deployment/app/)): `model.py` (tokenizer, model, `Prediktor`) dan `latih.py` (training). Di Colab, file-file itu diunduh otomatis dari GitHub.

In [ ]:
import os
import sys
import urllib.request
from pathlib import Path

QUICK_RUN = os.environ.get("QUICK_RUN") == "1"
os.environ["MLFLOW_DISABLE_AGENT_HINT"] = "1"          # sembunyikan pesan petunjuk MLflow
RAW = "https://raw.githubusercontent.com/riki-profile/ai-engineer-journey/main"

APP12 = Path("../12-deployment/app")                   # jika repo di-clone: pakai file lokal
if not (APP12 / "model.py").exists():                  # di Colab: unduh dari GitHub
    APP12 = Path("app12")
    APP12.mkdir(exist_ok=True)
    for f in ["model.py", "latih.py"]:
        urllib.request.urlretrieve(f"{RAW}/12-deployment/app/{f}", APP12 / f)
if not Path("pyfunc_sentimen.py").exists():
    urllib.request.urlretrieve(f"{RAW}/13-mlops/pyfunc_sentimen.py", "pyfunc_sentimen.py")
sys.path.insert(0, str(APP12.resolve()))

EPOCH = 3 if QUICK_RUN else 30
print(f"Kode modul 12 dari: {APP12.resolve()} | QUICK_RUN: {QUICK_RUN} | epoch: {EPOCH}")

## 1. Keacakan dan Variasi antar-Seed

Training melibatkan keacakan (inisialisasi bobot, urutan batch). Dengan **seed** yang sama, hasilnya harus identik (di hardware dan versi library yang sama). Dengan seed berbeda, hasilnya bervariasi, dan variasi inilah "noise" yang harus dikalahkan sebelum kita mengklaim suatu perubahan lebih baik.

In [ ]:
import pandas as pd

from latih import latih

a = latih("artefak_seed/a", epoch=EPOCH, seed=0, onnx=False)
b = latih("artefak_seed/b", epoch=EPOCH, seed=0, onnx=False)
print(f"Seed sama (0 dan 0): akurasi test {a['akurasi_test']} vs {b['akurasi_test']} → identik: {a['akurasi_test'] == b['akurasi_test']}")

SEEDS = [0, 1, 2] if QUICK_RUN else [0, 1, 2, 3, 4]
hasil_seed = pd.DataFrame([{"seed": s, **{k: v for k, v in latih(f"artefak_seed/s{s}", epoch=EPOCH, seed=s, onnx=False).items()
                                          if k.startswith("akurasi")}} for s in SEEDS])
print(hasil_seed.to_string(index=False))
print(f"\nAkurasi test: rata-rata {hasil_seed['akurasi_test'].mean():.3f} ± {hasil_seed['akurasi_test'].std():.3f} (simpangan baku)")

Jika dua konfigurasi di sweep notebook 01 berselisih lebih kecil dari simpangan baku ini, kita belum bisa menyimpulkan mana yang lebih baik. Praktik yang baik: laporkan **rata-rata ± simpangan baku dari beberapa seed**, terutama untuk data kecil.

Catatan: pada GPU, beberapa operasi tidak deterministik secara default. Gunakan `torch.use_deterministic_algorithms(True)` jika hasil harus identik bit per bit (dengan konsekuensi lebih lambat).

## 2. Versioning Data

Kode di-version dengan Git, tetapi dataset besar tidak cocok disimpan di Git. Dua gagasan penting:
1. **Sidik jari (hash)**: SHA-256 dari isi file. Satu byte berubah → hash berubah total. Simpan hash data di setiap run, sehingga kita tahu persis data mana yang dipakai.
2. **Penyimpanan terpisah**: file data disimpan di storage (S3, GCS, Google Drive), sedangkan Git hanya menyimpan file kecil berisi hash dan lokasinya. Inilah cara kerja **DVC** (*Data Version Control*):

```bash
dvc init
dvc add data/nusax              # membuat data/nusax.dvc berisi hash, file asli masuk .gitignore
git add data/nusax.dvc .gitignore && git commit -m "Data NusaX v1"
dvc remote add -d penyimpanan gdrive://<id-folder>   # atau s3://..., gs://...
dvc push                        # unggah data; rekan kerja cukup `git pull && dvc pull`
```

Kita terapkan versi sederhananya: unduh data, hitung hash, simpan *manifest*.

In [ ]:
import hashlib
import json

from latih import NUSAX_URL

DATA = Path("data/nusax")
DATA.mkdir(parents=True, exist_ok=True)
for split in ["train", "valid", "test"]:
    if not (DATA / f"{split}.csv").exists():
        urllib.request.urlretrieve(f"{NUSAX_URL}/{split}.csv", DATA / f"{split}.csv")


def sha256(path, ukuran_blok=1 << 20):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        while blok := f.read(ukuran_blok):
            h.update(blok)
    return h.hexdigest()


def buat_manifest(folder):
    return {f.name: {"sha256": sha256(f), "byte": f.stat().st_size} for f in sorted(Path(folder).glob("*.csv"))}


manifest = buat_manifest(DATA)
Path("manifest_data.json").write_text(json.dumps(manifest, indent=2))
hash_dataset = hashlib.sha256(json.dumps(manifest, sort_keys=True).encode()).hexdigest()
for nama, info in manifest.items():
    print(f"{nama:10} {info['sha256'][:16]}…  {info['byte']:,} byte")
print(f"Hash gabungan dataset: {hash_dataset[:16]}…")

Simulasi: seseorang "membersihkan" data train dengan menghapus satu baris. Manifest mendeteksinya:

In [ ]:
import shutil

salinan = Path("data/nusax_diubah")
shutil.copytree(DATA, salinan, dirs_exist_ok=True)
baris = (salinan / "train.csv").read_text(encoding="utf-8").splitlines(keepends=True)
(salinan / "train.csv").write_text("".join(baris[:-1]), encoding="utf-8")

manifest_baru = buat_manifest(salinan)
berubah = [nama for nama in manifest if manifest[nama]["sha256"] != manifest_baru[nama]["sha256"]]
print("File yang berubah:", berubah)

### Validasi data

Hash mendeteksi **perubahan**, tetapi tidak memberi tahu apakah datanya **benar**. **Validasi data** memeriksa aturan yang harus selalu dipenuhi (*data contract*) sebelum data dipakai training. Library seperti *Great Expectations* atau *Pandera* menyediakan ini secara lengkap; berikut versi sederhananya:

In [ ]:
LABEL_SAH = {"positive", "negative", "neutral"}


def periksa_data(df):
    """Kembalikan daftar masalah (kosong = lolos)."""
    masalah = []
    if missing := {"text", "label"} - set(df.columns):
        return [f"kolom hilang: {sorted(missing)}"]
    if n := df["text"].isna().sum() + (df["text"].astype(str).str.strip() == "").sum():
        masalah.append(f"{n} teks kosong")
    if n := df.duplicated("text").sum():
        masalah.append(f"{n} teks duplikat")
    if asing := set(df["label"].dropna()) - LABEL_SAH:
        masalah.append(f"label tidak dikenal: {sorted(asing)}")
    porsi = df["label"].value_counts(normalize=True)
    if porsi.min() < 0.10:
        masalah.append(f"kelas sangat tidak seimbang: {porsi.round(2).to_dict()}")
    if (panjang := df["text"].astype(str).str.len()).max() > 2000:
        masalah.append(f"{(panjang > 2000).sum()} teks lebih dari 2000 karakter")
    return masalah


train = pd.read_csv(DATA / "train.csv")
print("Data asli     :", periksa_data(train) or "✅ lolos")

rusak = pd.concat([train, train.head(3)]).reset_index(drop=True)          # duplikat
rusak.loc[0, "text"] = ""                                                 # teks kosong
rusak.loc[1, "label"] = "positif"                                         # label salah eja
print("Data bermasalah:", periksa_data(rusak))

## 3. Lineage: Mencatat Kode, Data, dan Environment

**Lineage** menjawab "model ini dibuat dari apa?". Setiap run MLflow sebaiknya mencatat:
- commit Git kode,
- hash data (manifest di atas),
- versi Python dan library penting, serta daftar lengkap paket (`pip freeze`) sebagai artefak.

In [ ]:
import importlib.metadata as meta
import platform
import subprocess

import mlflow
from mlflow import MlflowClient

mlflow.set_tracking_uri(f"sqlite:///{Path('mlflow.db').resolve()}")
mlflow.set_experiment("sentimen-reproducible")
client = MlflowClient()


def commit_git():
    r = subprocess.run(["git", "rev-parse", "HEAD"], capture_output=True, text=True)
    status = subprocess.run(["git", "status", "--porcelain"], capture_output=True, text=True)
    if r.returncode != 0:
        return "tidak diketahui"
    return r.stdout.strip() + ("-dirty" if status.stdout.strip() else "")   # dirty: ada perubahan belum di-commit


def info_environment():
    paket = ["torch", "numpy", "pandas", "onnx", "onnxruntime", "mlflow"]
    return {"python": platform.python_version(), "platform": platform.platform(),
            **{p: meta.version(p) for p in paket}}


with mlflow.start_run(run_name="run-reproducible") as run:
    config = latih("artefak_lineage", epoch=EPOCH, seed=0)
    mlflow.log_params({k: config[k] for k in ["dim", "lr", "epoch", "seed"]})
    mlflow.log_metrics({"akurasi_val": config["akurasi_val"], "akurasi_test": config["akurasi_test"]})
    mlflow.set_tags({"git_commit": commit_git(), "data_sha256": hash_dataset})
    mlflow.log_dict(info_environment(), "environment.json")
    mlflow.log_artifact("manifest_data.json")
    freeze = subprocess.run([sys.executable, "-m", "pip", "freeze"], capture_output=True, text=True).stdout
    mlflow.log_text(freeze, "requirements-lock.txt")

r = client.get_run(run.info.run_id)
print("Tag:", {k: v for k, v in r.data.tags.items() if not k.startswith("mlflow.")})
print("Artefak:", [a.path for a in client.list_artifacts(run.info.run_id)])
print("Environment:", mlflow.artifacts.load_dict(f"runs:/{run.info.run_id}/environment.json"))

Dengan catatan ini, mereproduksi model berarti: `git checkout <git_commit>`, ambil data dengan hash yang sama (`dvc pull`), pasang paket dari `requirements-lock.txt`, lalu jalankan training dengan parameter yang tercatat. Tag `-dirty` memperingatkan bahwa ada perubahan kode yang belum di-commit saat run dijalankan; run seperti ini tidak bisa direproduksi sepenuhnya.

## 4. Manajemen Environment

| Alat | Fungsi |
|---|---|
| `venv` + `requirements.txt` | Daftar paket (sering tanpa versi persis): cocok untuk belajar |
| **Lock file** (`pip freeze`, `pip-tools`, `uv lock`, `poetry.lock`) | Versi **persis** semua paket termasuk dependensi turunan: build yang sama setiap kali |
| **Docker** (modul 12) | Mengunci juga sistem operasi dan library sistem |
| `conda` | Berguna jika butuh library non-Python (CUDA, GDAL) |

Praktik yang baik: `requirements.txt` berisi dependensi langsung (untuk manusia), lock file dibuat otomatis dan di-commit (untuk mesin), lalu diperbarui secara berkala dengan sengaja (misalnya lewat PR otomatis Dependabot/Renovate) sambil menjalankan tes.

Catatan Colab: versi paket bawaan Colab berubah dari waktu ke waktu. Untuk hasil yang bisa diulang, pin versi di sel instalasi pertama, misalnya `%pip install torch==2.x.y`.

## 5. Uji Perilaku: Yang Tidak Terlihat dari Akurasi

Akurasi rata-rata bisa menyembunyikan kegagalan pada kasus yang sangat jelas. **Uji perilaku** (*behavioral testing*, gagasan dari makalah CheckList) memeriksa kemampuan spesifik dengan kasus buatan sendiri: kata sifat dasar, negasi, penguat, kalimat netral, dan sebagainya.

In [ ]:
from model import Prediktor

UJI_PERILAKU = [
    ("kata sifat positif", "Barangnya bagus sekali, saya sangat puas!", "positif"),
    ("kata sifat positif", "Makanannya enak dan pelayanannya ramah.", "positif"),
    ("kata sifat negatif", "Pelayanannya buruk sekali, saya kecewa.", "negatif"),
    ("kata sifat negatif", "Barangnya jelek dan rusak.", "negatif"),
    ("negasi", "Makanannya tidak enak.", "negatif"),
    ("negasi", "Pengirimannya tidak lambat sama sekali.", "positif"),
    ("netral/fakta", "Paket dikirim hari Senin lewat kurir.", "netral"),
    ("netral/fakta", "Toko ini buka jam sembilan pagi.", "netral"),
]
prediktor = Prediktor("artefak_lineage")
hasil_uji = pd.DataFrame([{"kemampuan": k, "teks": t, "harapan": h, "prediksi": p["label"],
                           "yakin": round(max(p["skor"].values()), 2), "lolos": p["label"] == h}
                          for (k, t, h), p in zip(UJI_PERILAKU, prediktor.prediksi([t for _, t, _ in UJI_PERILAKU]))])
print(hasil_uji.groupby("kemampuan")["lolos"].mean().round(2).to_dict())
hasil_uji

Model kecil ini (embedding + MLP tanpa urutan kata) mencapai ±70% akurasi di data test, tetapi biasanya gagal pada kasus seperti "jelek dan rusak" atau negasi ("tidak enak"), terkadang dengan keyakinan sangat tinggi. Model yang memperhatikan urutan kata (misalnya IndoBERT di modul 06) umumnya jauh lebih baik di sini.

Uji perilaku yang gagal bukan untuk disembunyikan: jadikan **daftar perbaikan** (tambah data, ganti model), dan kasus yang sudah benar dijadikan **uji regresi** agar tidak rusak di versi berikutnya.

## 6. CI/CD untuk ML

Selain kode, pipeline ML juga harus menguji **data** dan **model**:

| Tahap | Pemeriksaan | Di repo ini |
|---|---|---|
| **CI: kode** | Lint, unit test | `ruff`, `pytest` API (job *API & Docker*) |
| **CI: data** | Validasi skema & distribusi | `periksa_data` di atas (Latihan 2: jadikan tes) |
| **CI: model** | Latih cepat, **gerbang kualitas**: metrik ≥ ambang, uji regresi perilaku | [`tests/test_kualitas_model.py`](../12-deployment/app/tests/test_kualitas_model.py) |
| **CD** | Build image, deploy ke staging, *smoke test*, promosi | Build & uji container Docker di CI |
| **CT** (*continuous training*) | Latih ulang terjadwal atau saat drift terdeteksi | Notebook 03 |

Gerbang kualitas di repo ini: CI melatih model, lalu tes berikut **gagal** jika akurasi test di bawah ambang atau kasus regresi salah. Akibatnya, perubahan kode/data yang merusak model tidak akan lolos PR.

In [ ]:
tes_kualitas = APP12 / "tests" / "test_kualitas_model.py"
if tes_kualitas.exists():
    print(tes_kualitas.read_text())
    # Jalankan gerbang kualitas persis seperti di CI (memakai artefak modul 12; dilatih dulu bila belum ada).
    # Ambang diturunkan sedikit karena QUICK_RUN; di CI ambangnya 0.60.
    r = subprocess.run([sys.executable, "-m", "pytest", "-q", "tests/test_kualitas_model.py", "-p", "no:cacheprovider", "--color=no"],
                       cwd=APP12.resolve(), capture_output=True, text=True, env={**os.environ, "AMBANG_AKURASI": "0.55"})
    print(r.stdout[-800:], r.stderr[-800:])
else:
    print("File tes tidak tersedia (misalnya di Colab tanpa clone repo). Lihat di GitHub: 12-deployment/app/tests/")

Cuplikan job CI yang menjalankannya (dari [`.github/workflows/ci.yml`](../.github/workflows/ci.yml)):

```yaml
- name: Latih model & buat artefak
  run: python latih.py --epoch 10
- name: Tes API (pytest)          # termasuk tests/test_kualitas_model.py (gerbang kualitas)
  run: pytest -q
- name: Build image Docker
  run: docker build -t api-sentimen .
```

## Ringkasan

- Reproducibility butuh versi **kode, data, environment, seed** (dan catatan hardware).
- Laporkan **rata-rata ± simpangan baku** beberapa seed; selisih di bawah noise bukan perbaikan.
- **Versioning data** dengan hash/manifest atau DVC; **validasi data** sebelum training.
- Catat **lineage** di setiap run: commit git (tandai `-dirty`), hash data, environment, lock file.
- **Uji perilaku** menemukan kegagalan yang tersembunyi di balik akurasi rata-rata.
- **CI/CD untuk ML** menambahkan pengujian data dan model; **gerbang kualitas** mencegah model buruk ter-deploy.

## Latihan

1. **Reproduksi dari run.** Tulis fungsi `reproduksi(run_id)` yang membaca parameter dan tag run MLflow, memeriksa hash data saat ini sama dengan `data_sha256`, melatih ulang dengan parameter yang sama, lalu membandingkan akurasinya dengan yang tercatat.
2. **Validasi data sebagai tes.** Pindahkan `periksa_data` ke `12-deployment/app/validasi_data.py` dan tambahkan tes pytest yang memastikan data NusaX lolos validasi. Mengapa sebaiknya validasi dijalankan **sebelum** training di pipeline CI?
3. **Perbaiki kegagalan perilaku.** Tambahkan 50 kalimat berlabel buatanmu sendiri yang berisi negasi dan kata sifat negatif ke data train (simpan terpisah, catat hash-nya), latih ulang, dan ukur lagi uji perilaku **dan** akurasi test. Apakah perbaikan di satu sisi merusak sisi lain?